# 09. Diseno experimental de Semana 3

Este notebook formaliza el montaje experimental inicial para la fase de modelos del proyecto.

## Objetivos
- dejar definida la base oficial de modelado de esta semana
- especificar el target principal y el target complementario
- definir que variables entran, cuales se excluyen y por que
- fijar el split temporal de entrenamiento y prueba
- definir las metricas que se usaran en los baseline
- dejar lista la estructura de reporte para los notebooks de modelos


## Alineacion con el entregable de la semana

De acuerdo con las pautas de implementacion y pruebas de modelos revisadas para Semana 3, en esta etapa se espera:

- implementar los modelos a probar o comparar
- documentar el tratamiento previo de los datos
- definir el montaje experimental
- validar ese montaje a pequena escala
- comprobar que el proceso capture y reporte la informacion necesaria

Por eso este notebook no busca entrenar todavia todos los modelos posibles. Su funcion es dejar cerrada la metodologia minima y reproducible para que los baseline siguientes corran con un criterio consistente.


## Comentario metodologico clave

La capa principal de modelado de esta fase sera anual, a nivel `departamento-anio`. La base mensual sigue siendo muy valiosa, pero su uso sera complementario.

- La base anual contiene el target principal mejor sustentado metodologicamente.
- La base mensual sirve para estacionalidad, soporte del informe tecnico, dashboard y construccion de agregados.
- En consecuencia, los modelos baseline de Semana 3 se montaran sobre la base anual final limpia.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 180)


def find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'BASE_DE_DATOS').exists():
            return candidate
    raise FileNotFoundError('No se encontro una carpeta BASE_DE_DATOS en la ruta actual ni en sus padres.')


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
BASE_DATOS = PROJECT_ROOT / 'BASE_DE_DATOS'
ANNUAL_PATH = BASE_DATOS / 'FINALES' / 'dataset_modelado_anual_limpio.csv'
MONTHLY_PATH = BASE_DATOS / 'FINALES' / 'dataset_operativo_mensual_limpio.csv'


In [2]:
for path in [ANNUAL_PATH, MONTHLY_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'No existe el archivo esperado: {path}')

print('Rutas validadas correctamente.')
print('Base anual:', ANNUAL_PATH.name)
print('Base mensual complementaria:', MONTHLY_PATH.name)


Rutas validadas correctamente.
Base anual: dataset_modelado_anual_limpio.csv
Base mensual complementaria: dataset_operativo_mensual_limpio.csv


## Carga de bases y seleccion de la base oficial de modelado

Aunque aqui se cargan ambas bases para efectos de trazabilidad, la base oficial del montaje experimental de Semana 3 sera la anual.


In [3]:
annual = pd.read_csv(ANNUAL_PATH, sep=';').sort_values(['departamento', 'anio']).reset_index(drop=True)
monthly = pd.read_csv(MONTHLY_PATH, sep=';').sort_values(['departamento', 'anio', 'mes']).reset_index(drop=True)

annual['es_risaralda'] = annual['departamento'].eq('Risaralda').astype(int)

base_choice = pd.DataFrame([
    {'base': 'dataset_modelado_anual_limpio.csv', 'rol': 'base principal de modelado', 'unidad_analisis': 'departamento-anio', 'justificacion': 'contiene el target anual mejor sustentado y la capa principal de variables agregadas'},
    {'base': 'dataset_operativo_mensual_limpio.csv', 'rol': 'base complementaria', 'unidad_analisis': 'departamento-mes', 'justificacion': 'sirve para lectura estacional, operativa y descriptiva, pero no reemplaza el target anual observado'}
])

print('Shape base anual:', annual.shape)
print('Shape base mensual:', monthly.shape)
display(base_choice)


Shape base anual: (36, 81)
Shape base mensual: (628, 60)


,base,rol,unidad_analisis,justificacion
0,dataset_modelado_anual_limpio.csv,base principal de modelado,departamento-anio,contiene el target anual mejor sustentado y la capa principal de variables agregadas
1,dataset_operativo_mensual_limpio.csv,base complementaria,departamento-mes,"sirve para lectura estacional, operativa y descriptiva, pero no reemplaza el target anual observado"


## Definicion del problema analitico

En esta semana se dejan activas dos formulaciones complementarias:

- una formulacion principal de regresion para estimar la magnitud de la perdida anual
- una formulacion complementaria de clasificacion para identificar eventos de perdida anual


In [4]:
problem_definition = pd.DataFrame([
    {'componente': 'unidad_analisis', 'definicion': 'departamento-anio'},
    {'componente': 'problema_principal', 'definicion': 'regresion'},
    {'componente': 'problema_complementario', 'definicion': 'clasificacion binaria'},
    {'componente': 'target_regresion', 'definicion': 'perdida_rendimiento_anual_pct'},
    {'componente': 'target_clasificacion', 'definicion': 'evento_perdida_anual'},
    {'componente': 'base_oficial_semana_3', 'definicion': 'dataset_modelado_anual_limpio.csv'}
])
display(problem_definition)


,componente,definicion
0,unidad_analisis,departamento-anio
1,problema_principal,regresion
2,problema_complementario,clasificacion binaria
3,target_regresion,perdida_rendimiento_anual_pct
4,target_clasificacion,evento_perdida_anual
5,base_oficial_semana_3,dataset_modelado_anual_limpio.csv


## Trazabilidad entre requerimientos, capas analiticas y salidas del proyecto

La fase de modelado no se plantea como un ejercicio aislado. Cada capa analitica responde a un requerimiento concreto del proyecto y debe conectarse con una salida util para el prototipo.

- La capa descriptiva resume comportamiento productivo, climatico y satelital, y ayuda a interpretar el problema.
- La capa predictiva anual estima magnitud de perdida y permite comparar modelos de forma metodologicamente defendible.
- La capa operacional mensual traduce esa logica a una senal mas cercana al trigger, al dashboard y al monitoreo continuo.
- La capa prescriptiva se apoyara despues en estos resultados para ajustar umbrales, cobertura y decisiones de activacion.


In [5]:
requirement_traceability = pd.DataFrame([
    {
        'requerimiento_proyecto': 'Estimar perdida relevante para el seguro',
        'pregunta_analitica': 'Cuanto cae el rendimiento anual esperado?',
        'capa_analitica': 'predictiva principal',
        'base_soporte': 'anual',
        'tecnicas_a_probar': 'regresion lineal, regularizados, arboles, boosting, redes neuronales',
        'salida_esperada': 'prediccion continua de perdida anual',
        'conexion_prototipo': 'alimenta comparacion de modelos, basis risk y simulador'
    },
    {
        'requerimiento_proyecto': 'Detectar eventos de perdida anual',
        'pregunta_analitica': 'Se puede identificar un anio critico con recall aceptable?',
        'capa_analitica': 'predictiva complementaria',
        'base_soporte': 'anual',
        'tecnicas_a_probar': 'regresion con lectura de eventos y clasificacion dedicada en notebook 11',
        'salida_esperada': 'evento anual y score de severidad',
        'conexion_prototipo': 'apoya trigger, cobertura y reduccion del basis risk'
    },
    {
        'requerimiento_proyecto': 'Monitorear riesgo intra-anual para el dashboard',
        'pregunta_analitica': 'Que senal mensual ayuda a anticipar anos malos o meses criticos?',
        'capa_analitica': 'operacional mensual',
        'base_soporte': 'mensual',
        'tecnicas_a_probar': 'modelos operativos con rezagos, score mensual y evaluacion anualizada',
        'salida_esperada': 'score operacional mensual por departamento',
        'conexion_prototipo': 'alimenta dashboard, alertas y estado del trigger'
    },
    {
        'requerimiento_proyecto': 'Tomar decisiones de umbral y cobertura',
        'pregunta_analitica': 'Como balancear error continuo, recall de eventos y costo operativo?',
        'capa_analitica': 'prescriptiva futura',
        'base_soporte': 'anual + mensual',
        'tecnicas_a_probar': 'comparacion de metricas, reglas de negocio y simulacion',
        'salida_esperada': 'criterios para trigger y simulador de cobertura',
        'conexion_prototipo': 'conecta analitica con decisiones del artefacto final'
    },
])

integration_layers = pd.DataFrame([
    {'capa': 'descriptiva', 'rol': 'entender cobertura, estacionalidad y calidad de la senal', 'producto': 'EDA anual y mensual', 'estado_en_el_proyecto': 'ya implementada'},
    {'capa': 'predictiva anual', 'rol': 'validacion principal del target y comparacion de modelos', 'producto': 'modelo anual y ranking de desempeno', 'estado_en_el_proyecto': 'fase activa en notebooks 10 y 11'},
    {'capa': 'operacional mensual', 'rol': 'traducir la logica predictiva a seguimiento intra-anual', 'producto': 'score operacional mensual', 'estado_en_el_proyecto': 'fase activa en notebook 10.1'},
    {'capa': 'prescriptiva', 'rol': 'ajustar umbrales, cobertura y decisiones', 'producto': 'simulador y reglas de activacion', 'estado_en_el_proyecto': 'pendiente de consolidar'}
])

display(requirement_traceability)
display(integration_layers)


,requerimiento_proyecto,pregunta_analitica,capa_analitica,base_soporte,tecnicas_a_probar,salida_esperada,conexion_prototipo
0,Estimar perdida relevante para el seguro,Cuanto cae el rendimiento anual esperado?,predictiva principal,anual,"regresion lineal, regularizados, arboles, boosting, redes neuronales",prediccion continua de perdida anual,"alimenta comparacion de modelos, basis risk y simulador"
1,Detectar eventos de perdida anual,Se puede identificar un anio critico con recall aceptable?,predictiva complementaria,anual,regresion con lectura de eventos y clasificacion dedicada en notebook 11,evento anual y score de severidad,"apoya trigger, cobertura y reduccion del basis risk"
2,Monitorear riesgo intra-anual para el dashboard,Que senal mensual ayuda a anticipar anos malos o meses criticos?,operacional mensual,mensual,"modelos operativos con rezagos, score mensual y evaluacion anualizada",score operacional mensual por departamento,"alimenta dashboard, alertas y estado del trigger"
3,Tomar decisiones de umbral y cobertura,"Como balancear error continuo, recall de eventos y costo operativo?",prescriptiva futura,anual + mensual,"comparacion de metricas, reglas de negocio y simulacion",criterios para trigger y simulador de cobertura,conecta analitica con decisiones del artefacto final


,capa,rol,producto,estado_en_el_proyecto
0,descriptiva,"entender cobertura, estacionalidad y calidad de la senal",EDA anual y mensual,ya implementada
1,predictiva anual,validacion principal del target y comparacion de modelos,modelo anual y ranking de desempeno,fase activa en notebooks 10 y 11
2,operacional mensual,traducir la logica predictiva a seguimiento intra-anual,score operacional mensual,fase activa en notebook 10.1
3,prescriptiva,"ajustar umbrales, cobertura y decisiones",simulador y reglas de activacion,pendiente de consolidar


## Targets y distribucion basica

Antes de definir variables y pruebas, verificamos la naturaleza de los targets que vamos a usar.


In [6]:
target_reg = 'perdida_rendimiento_anual_pct'
target_clf = 'evento_perdida_anual'

target_summary = pd.DataFrame([
    {'target': target_reg, 'tipo': 'continuo', 'nulos': int(annual[target_reg].isna().sum()), 'min': float(annual[target_reg].min()), 'max': float(annual[target_reg].max()), 'media': float(annual[target_reg].mean())},
    {'target': target_clf, 'tipo': 'binario', 'nulos': int(annual[target_clf].isna().sum()), 'min': int(annual[target_clf].min()), 'max': int(annual[target_clf].max()), 'media': float(annual[target_clf].mean())}
])

class_balance = annual[target_clf].value_counts(dropna=False).rename_axis('clase').reset_index(name='n')
class_balance['proporcion'] = class_balance['n'] / class_balance['n'].sum()

display(target_summary)
display(class_balance)


,target,tipo,nulos,min,max,media
0,perdida_rendimiento_anual_pct,continuo,0,-31.525406,28.904244,1.268121e-14
1,evento_perdida_anual,binario,0,0.000000,1.000000,1.944444e-01


,clase,n,proporcion
0,0,29,0.805556
1,1,7,0.194444


## Variables excluidas del modelado baseline

No todas las columnas de la base anual deben entrar al baseline. Algunas se excluyen porque son:

- el propio target o una reformulacion directa del target
- variables de resultado observadas que inducen leakage ex post
- banderas o metadatos de limpieza
- columnas auxiliares de trazabilidad y no de prediccion


In [7]:
id_cols = ['departamento', 'anio']

target_and_direct_leakage = [
    'perdida_rendimiento_anual_pct',
    'perdida_rendimiento_anual_pct_prev',
    'perdida_rendimiento_anual_pct_prev3',
    'evento_perdida_anual',
    'perdida_real_pct',
    'evento_perdida_real',
    'diff_target_vs_perdida_real_pct'
]

current_outcome_or_post_event = [
    'rendimiento_t_ha',
    'produccion_t',
    'rendimiento_referencia_fullsample',
    'rendimiento_referencia_prev',
    'rendimiento_referencia_prev3',
    'produccion_t_original',
    'rendimiento_t_ha_original',
    'delta_produccion_t',
    'delta_rendimiento_t_ha',
    'rendimiento_medio_municipal_reportado',
    'dif_rendimiento_calculado_vs_reportado',
    'rendimiento_medio_t_ha',
    'produccion_media_t'
]

quality_or_traceability = [
    'correccion_aplicada',
    'motivo_correccion',
    'disponible_referencia_prev',
    'disponible_referencia_prev3',
    'listo_para_modelado_principal',
    'version_target_principal'
]

baseline_excluded = id_cols + target_and_direct_leakage + current_outcome_or_post_event + quality_or_traceability
baseline_excluded = [c for c in baseline_excluded if c in annual.columns]

exclusion_table = pd.DataFrame([
    {'grupo_exclusion': 'identificacion', 'n_variables': len(id_cols), 'variables': ', '.join(id_cols)},
    {'grupo_exclusion': 'target_y_leakage_directo', 'n_variables': len(target_and_direct_leakage), 'variables': ', '.join(target_and_direct_leakage)},
    {'grupo_exclusion': 'resultado_observado_ex_post', 'n_variables': len(current_outcome_or_post_event), 'variables': ', '.join(current_outcome_or_post_event)},
    {'grupo_exclusion': 'calidad_y_trazabilidad', 'n_variables': len(quality_or_traceability), 'variables': ', '.join(quality_or_traceability)}
])

display(exclusion_table)


,grupo_exclusion,n_variables,variables
0,identificacion,2,"departamento, anio"
1,target_y_leakage_directo,7,"perdida_rendimiento_anual_pct, perdida_rendimiento_anual_pct_prev, perdida_rendimiento_anual_pct_prev3, evento_perdida_anual, perdida_real_pct, evento_perdida_real, diff_target..."
2,resultado_observado_ex_post,13,"rendimiento_t_ha, produccion_t, rendimiento_referencia_fullsample, rendimiento_referencia_prev, rendimiento_referencia_prev3, produccion_t_original, rendimiento_t_ha_original, ..."
3,calidad_y_trazabilidad,6,"correccion_aplicada, motivo_correccion, disponible_referencia_prev, disponible_referencia_prev3, listo_para_modelado_principal, version_target_principal"


## Variables candidatas y conjuntos de features

Dado el tamano pequeno de la muestra, conviene diferenciar entre:

- un pool amplio de variables candidatas
- un conjunto parsimonioso para los baseline de Semana 3

El conjunto parsimonioso prioriza interpretabilidad, estabilidad y una relacion razonable entre numero de observaciones y numero de variables.


In [8]:
baseline_feature_set = [
    'es_risaralda',
    'precio_ico_usd_ton',
    'precipitation_annual_sum',
    'temp_aire_C_annual_mean',
    'def_annual_mean',
    'GDD_cafe_annual_mean',
    'NDVI_anomalia_pct_annual_mean',
    'precipitation_cosecha_sum',
    'temp_aire_C_cosecha_mean',
    'NDVI_anomalia_pct_cosecha_mean'
]

exploratory_feature_pool = [
    'es_risaralda',
    'n_municipios',
    'precio_ico_usd_ton',
    'precio_productor_usd_ton',
    'precipitation_annual_sum',
    'temp_aire_C_annual_mean',
    'humedad_relativa_pct_annual_mean',
    'soil_annual_mean',
    'def_annual_mean',
    'pet_annual_mean',
    'aet_annual_mean',
    'GDD_cafe_annual_mean',
    'NDVI_annual_mean',
    'EVI_annual_mean',
    'Gpp_annual_mean',
    'NDVI_anomalia_pct_annual_mean',
    'EVI_anomalia_pct_annual_mean',
    'Gpp_anomalia_pct_annual_mean',
    'precipitation_anomalia_pct_annual_mean',
    'precipitation_cosecha_sum',
    'temp_aire_C_cosecha_mean',
    'humedad_relativa_pct_cosecha_mean',
    'def_cosecha_mean',
    'GDD_cafe_cosecha_mean',
    'NDVI_cosecha_mean',
    'EVI_cosecha_mean',
    'Gpp_cosecha_mean',
    'NDVI_anomalia_pct_cosecha_mean',
    'EVI_anomalia_pct_cosecha_mean',
    'Gpp_anomalia_pct_cosecha_mean',
    'elevacion_media_m',
    'pendiente_media'
]

missing_baseline = [c for c in baseline_feature_set if c not in annual.columns]
missing_pool = [c for c in exploratory_feature_pool if c not in annual.columns]
assert not missing_baseline, f'Faltan columnas del baseline: {missing_baseline}'
assert not missing_pool, f'Faltan columnas del pool exploratorio: {missing_pool}'

feature_sets = pd.DataFrame([
    {'conjunto': 'baseline_parsimonioso', 'n_variables': len(baseline_feature_set), 'variables': ', '.join(baseline_feature_set)},
    {'conjunto': 'pool_exploratorio', 'n_variables': len(exploratory_feature_pool), 'variables': ', '.join(exploratory_feature_pool)}
])

display(feature_sets)


,conjunto,n_variables,variables
0,baseline_parsimonioso,10,"es_risaralda, precio_ico_usd_ton, precipitation_annual_sum, temp_aire_C_annual_mean, def_annual_mean, GDD_cafe_annual_mean, NDVI_anomalia_pct_annual_mean, precipitation_cosecha..."
1,pool_exploratorio,32,"es_risaralda, n_municipios, precio_ico_usd_ton, precio_productor_usd_ton, precipitation_annual_sum, temp_aire_C_annual_mean, humedad_relativa_pct_annual_mean, soil_annual_mean,..."


## Supuestos relevantes por familia de modelos y tratamiento previo esperado

La comparacion de modelos no parte de la idea de que todos exigen exactamente el mismo tratamiento. Aunque varias familias comparten la misma base de entrada, cada una trae supuestos y riesgos diferentes.

- Los modelos lineales y regularizados suponen relaciones mas cercanas a linealidad y son sensibles a colinealidad, escala y outliers.
- Los modelos basados en arboles y boosting son mas flexibles respecto a forma funcional, pero pueden sobreajustarse si la validacion temporal es debil.
- Las redes neuronales requieren especial cuidado con escala, tamano muestral y estabilidad de entrenamiento.
- La capa mensual agrega ademas el riesgo de leakage temporal y la necesidad de representar memoria reciente mediante rezagos de forma controlada.


In [9]:
assumptions_and_preprocessing = pd.DataFrame([
    {
        'familia_modelo': 'lineal y regularizado',
        'supuestos_o_riesgos': 'relaciones aproximadamente lineales, sensibilidad a colinealidad y a valores influyentes',
        'tratamiento_previo': 'excluir leakage, usar feature sets parsimoniosos y considerar escalamiento en notebooks de entrenamiento',
        'evidencia_o_verificacion': 'EDA anual, correlaciones iniciales, seleccion manual y automatica de variables, smoke test',
        'riesgo_remanente': 'la linealidad perfecta no esta garantizada por el problema'
    },
    {
        'familia_modelo': 'lineal robusto',
        'supuestos_o_riesgos': 'busca reducir sensibilidad a outliers, pero sigue dependiendo de senal estable y muestra pequena',
        'tratamiento_previo': 'mantener version limpia del target anual y splits temporales estrictos',
        'evidencia_o_verificacion': 'correccion de Cundinamarca 2008, control de filas extremas y revision de dispersion del target',
        'riesgo_remanente': 'puede perder eficiencia si la senal no es aproximadamente lineal'
    },
    {
        'familia_modelo': 'dimension reducida',
        'supuestos_o_riesgos': 'resume informacion correlacionada, pero reduce interpretabilidad directa',
        'tratamiento_previo': 'seleccionar bloques coherentes de variables y evitar columnas redundantes por leakage',
        'evidencia_o_verificacion': 'EDA anual mostro colinealidad entre variables satelitales y climaticas',
        'riesgo_remanente': 'componentes pueden ser estables estadisticamente, pero mas dificiles de explicar al usuario'
    },
    {
        'familia_modelo': 'gam y gam_like',
        'supuestos_o_riesgos': 'permiten no linealidad suave, pero requieren suficiente senal para no sobredimensionar patrones',
        'tratamiento_previo': 'mantener pocos predictores y validar siempre con split temporal',
        'evidencia_o_verificacion': 'se incluyen como prueba de relaciones no lineales mas interpretables que boosting puro',
        'riesgo_remanente': 'muestra pequena puede limitar el valor real del suavizado'
    },
    {
        'familia_modelo': 'arboles y boosting',
        'supuestos_o_riesgos': 'menos dependencia de linealidad, pero riesgo de sobreajuste y soluciones degeneradas si no se controla el entrenamiento',
        'tratamiento_previo': 'validacion temporal expansiva, ranking con penalizacion a predicciones degeneradas y comparacion holdout',
        'evidencia_o_verificacion': 'reescritura del notebook 10 para detectar predicciones casi constantes y revisar variabilidad predicha',
        'riesgo_remanente': 'pueden ganar en RMSE y aun asi no detectar eventos severos'
    },
    {
        'familia_modelo': 'redes neuronales',
        'supuestos_o_riesgos': 'alta flexibilidad, pero requieren escala adecuada y son sensibles al tamano muestral',
        'tratamiento_previo': 'usar conjuntos de variables controlados, tuning moderado y lectura cuidadosa de estabilidad entre CV y holdout',
        'evidencia_o_verificacion': 'se prueban como exploracion de complejidad adicional, no como baseline de interpretabilidad',
        'riesgo_remanente': 'pueden mostrar sobreajuste o variacion alta en muestra pequena'
    },
    {
        'familia_modelo': 'operacional mensual',
        'supuestos_o_riesgos': 'no existe un target mensual observado equivalente al anual; riesgo de leakage temporal y de sobreinterpretar proxies',
        'tratamiento_previo': 'usar la capa mensual como complemento, agregar rezagos y evaluar con anualizacion posterior frente al target anual observado',
        'evidencia_o_verificacion': 'decision metodologica explicita de no reemplazar la validacion anual por la mensual',
        'riesgo_remanente': 'aporta operacion y trigger, pero no sustituye la verdad principal del problema'
    },
])

display(assumptions_and_preprocessing)


,familia_modelo,supuestos_o_riesgos,tratamiento_previo,evidencia_o_verificacion,riesgo_remanente
0,lineal y regularizado,"relaciones aproximadamente lineales, sensibilidad a colinealidad y a valores influyentes","excluir leakage, usar feature sets parsimoniosos y considerar escalamiento en notebooks de entrenamiento","EDA anual, correlaciones iniciales, seleccion manual y automatica de variables, smoke test",la linealidad perfecta no esta garantizada por el problema
1,lineal robusto,"busca reducir sensibilidad a outliers, pero sigue dependiendo de senal estable y muestra pequena",mantener version limpia del target anual y splits temporales estrictos,"correccion de Cundinamarca 2008, control de filas extremas y revision de dispersion del target",puede perder eficiencia si la senal no es aproximadamente lineal
2,dimension reducida,"resume informacion correlacionada, pero reduce interpretabilidad directa",seleccionar bloques coherentes de variables y evitar columnas redundantes por leakage,EDA anual mostro colinealidad entre variables satelitales y climaticas,"componentes pueden ser estables estadisticamente, pero mas dificiles de explicar al usuario"
3,gam y gam_like,"permiten no linealidad suave, pero requieren suficiente senal para no sobredimensionar patrones",mantener pocos predictores y validar siempre con split temporal,se incluyen como prueba de relaciones no lineales mas interpretables que boosting puro,muestra pequena puede limitar el valor real del suavizado
4,arboles y boosting,"menos dependencia de linealidad, pero riesgo de sobreajuste y soluciones degeneradas si no se controla el entrenamiento","validacion temporal expansiva, ranking con penalizacion a predicciones degeneradas y comparacion holdout",reescritura del notebook 10 para detectar predicciones casi constantes y revisar variabilidad predicha,pueden ganar en RMSE y aun asi no detectar eventos severos
5,redes neuronales,"alta flexibilidad, pero requieren escala adecuada y son sensibles al tamano muestral","usar conjuntos de variables controlados, tuning moderado y lectura cuidadosa de estabilidad entre CV y holdout","se prueban como exploracion de complejidad adicional, no como baseline de interpretabilidad",pueden mostrar sobreajuste o variacion alta en muestra pequena
6,operacional mensual,no existe un target mensual observado equivalente al anual; riesgo de leakage temporal y de sobreinterpretar proxies,"usar la capa mensual como complemento, agregar rezagos y evaluar con anualizacion posterior frente al target anual observado",decision metodologica explicita de no reemplazar la validacion anual por la mensual,"aporta operacion y trigger, pero no sustituye la verdad principal del problema"


## Validacion de nulos en las variables seleccionadas

Antes de definir el split, confirmamos que el baseline parsimonioso no tenga faltantes inesperados.


In [10]:
baseline_nulls = annual[baseline_feature_set + [target_reg, target_clf]].isna().sum().rename('nulos').reset_index().rename(columns={'index': 'variable'})
display(baseline_nulls)
assert baseline_nulls['nulos'].sum() == 0, 'El conjunto baseline tiene faltantes y debe revisarse antes del modelado.'


,variable,nulos
0,es_risaralda,0
1,precio_ico_usd_ton,0
2,precipitation_annual_sum,0
3,temp_aire_C_annual_mean,0
4,def_annual_mean,0
5,GDD_cafe_annual_mean,0
6,NDVI_anomalia_pct_annual_mean,0
7,precipitation_cosecha_sum,0
8,temp_aire_C_cosecha_mean,0
9,NDVI_anomalia_pct_cosecha_mean,0


## Definicion del split temporal

Como se trata de un problema con dimension temporal y muy pocas observaciones, no conviene usar un split aleatorio.

Para Semana 3 se definiran dos escalas de trabajo:

- un split principal train-test para los baseline
- un split pequeno de desarrollo para comprobar que el pipeline experimental reporte bien la informacion


In [11]:
TRAIN_END_YEAR = 2020
SMALL_TRAIN_END_YEAR = 2018
SMALL_VALID_END_YEAR = 2020

annual['split_principal'] = np.where(annual['anio'] <= TRAIN_END_YEAR, 'train', 'test')
annual['split_pequeno'] = np.select(
    [annual['anio'] <= SMALL_TRAIN_END_YEAR, annual['anio'] <= SMALL_VALID_END_YEAR],
    ['train_small', 'valid_small'],
    default='test_small'
)

split_main_summary = annual.groupby(['split_principal', 'departamento']).size().reset_index(name='n')
split_small_summary = annual.groupby(['split_pequeno', 'departamento']).size().reset_index(name='n')
split_years = annual[['anio', 'split_principal', 'split_pequeno']].drop_duplicates().sort_values('anio').reset_index(drop=True)

display(split_main_summary)
display(split_small_summary)
display(split_years)

assert annual.loc[annual['split_principal'] == 'train', 'anio'].max() == TRAIN_END_YEAR
assert annual.loc[annual['split_principal'] == 'test', 'anio'].min() == TRAIN_END_YEAR + 1


,split_principal,departamento,n
0,test,Cundinamarca,4
1,test,Risaralda,4
2,train,Cundinamarca,14
3,train,Risaralda,14


,split_pequeno,departamento,n
0,test_small,Cundinamarca,4
1,test_small,Risaralda,4
2,train_small,Cundinamarca,12
3,train_small,Risaralda,12
4,valid_small,Cundinamarca,2
5,valid_small,Risaralda,2


,anio,split_principal,split_pequeno
0,2007,train,train_small
1,2008,train,train_small
2,2009,train,train_small
3,2010,train,train_small
4,2011,train,train_small
5,2012,train,train_small
6,2013,train,train_small
7,2014,train,train_small
8,2015,train,train_small
9,2016,train,train_small


## Metricas de evaluacion

Las metricas elegidas responden tanto a las pautas del curso como a la naturaleza del problema.

- Regresion: interesa medir magnitud del error y capacidad explicativa general.
- Clasificacion: interesa especialmente no perder eventos reales de perdida, por lo que recall tendra una lectura prioritaria.


In [12]:
regression_metrics = pd.DataFrame([
    {'metrica': 'MAE', 'lectura': 'mide error absoluto promedio', 'prioridad': 'alta'},
    {'metrica': 'RMSE', 'lectura': 'penaliza mas los errores grandes', 'prioridad': 'alta'},
    {'metrica': 'R2', 'lectura': 'mide ajuste general del modelo', 'prioridad': 'media'}
])

classification_metrics = pd.DataFrame([
    {'metrica': 'Accuracy', 'lectura': 'referencia general, pero no suficiente por si sola', 'prioridad': 'media'},
    {'metrica': 'Precision', 'lectura': 'importa si se quiere evitar falsas alarmas', 'prioridad': 'media'},
    {'metrica': 'Recall', 'lectura': 'prioritaria para no perder eventos reales de perdida', 'prioridad': 'alta'},
    {'metrica': 'F1', 'lectura': 'balance entre precision y recall', 'prioridad': 'alta'},
    {'metrica': 'Matriz de confusion', 'lectura': 'muestra la estructura de errores del clasificador', 'prioridad': 'alta'}
])

display(regression_metrics)
display(classification_metrics)


,metrica,lectura,prioridad
0,MAE,mide error absoluto promedio,alta
1,RMSE,penaliza mas los errores grandes,alta
2,R2,mide ajuste general del modelo,media


,metrica,lectura,prioridad
0,Accuracy,"referencia general, pero no suficiente por si sola",media
1,Precision,importa si se quiere evitar falsas alarmas,media
2,Recall,prioritaria para no perder eventos reales de perdida,alta
3,F1,balance entre precision y recall,alta
4,Matriz de confusion,muestra la estructura de errores del clasificador,alta


## Estructura de reporte para los experimentos

Los notebooks 10 y 11 deberian producir resultados compatibles con una tabla minima de reporte como la siguiente.


In [13]:
experiment_report_template = pd.DataFrame([
    {
        'nombre_experimento': 'baseline_regresion_lineal',
        'tipo_modelo': 'regresion',
        'dataset': 'dataset_modelado_anual_limpio.csv',
        'target': 'perdida_rendimiento_anual_pct',
        'feature_set': 'baseline_parsimonioso',
        'train_periodo': '2007-2020',
        'test_periodo': '2021-2024',
        'metricas_reportadas': 'MAE, RMSE, R2',
        'observacion': 'comprobar que el pipeline capture valores reales, predichos y metricas'
    },
    {
        'nombre_experimento': 'baseline_regresion_logistica',
        'tipo_modelo': 'clasificacion',
        'dataset': 'dataset_modelado_anual_limpio.csv',
        'target': 'evento_perdida_anual',
        'feature_set': 'baseline_parsimonioso',
        'train_periodo': '2007-2020',
        'test_periodo': '2021-2024',
        'metricas_reportadas': 'Accuracy, Precision, Recall, F1, matriz de confusion',
        'observacion': 'evaluar especialmente recall por la importancia de detectar eventos'
    }
])
display(experiment_report_template)


,nombre_experimento,tipo_modelo,dataset,target,feature_set,train_periodo,test_periodo,metricas_reportadas,observacion
0,baseline_regresion_lineal,regresion,dataset_modelado_anual_limpio.csv,perdida_rendimiento_anual_pct,baseline_parsimonioso,2007-2020,2021-2024,"MAE, RMSE, R2","comprobar que el pipeline capture valores reales, predichos y metricas"
1,baseline_regresion_logistica,clasificacion,dataset_modelado_anual_limpio.csv,evento_perdida_anual,baseline_parsimonioso,2007-2020,2021-2024,"Accuracy, Precision, Recall, F1, matriz de confusion",evaluar especialmente recall por la importancia de detectar eventos


## Tabla consolidada del montaje experimental

Esta tabla resume lo que debe quedar entendido y fijado al cerrar este notebook.


In [14]:
montaje_experimental = pd.DataFrame([
    {'componente': 'base principal', 'decision': 'dataset_modelado_anual_limpio.csv'},
    {'componente': 'base complementaria', 'decision': 'dataset_operativo_mensual_limpio.csv'},
    {'componente': 'unidad de analisis', 'decision': 'departamento-anio'},
    {'componente': 'target principal', 'decision': 'perdida_rendimiento_anual_pct'},
    {'componente': 'target complementario', 'decision': 'evento_perdida_anual'},
    {'componente': 'feature set baseline', 'decision': '10 variables parsimoniosas'},
    {'componente': 'split principal', 'decision': 'train 2007-2020 | test 2021-2024'},
    {'componente': 'split pequeno de verificacion', 'decision': 'train 2007-2018 | valid 2019-2020 | test 2021-2024'},
    {'componente': 'metricas regresion', 'decision': 'MAE, RMSE, R2'},
    {'componente': 'metricas clasificacion', 'decision': 'Accuracy, Precision, Recall, F1, matriz de confusion'},
    {'componente': 'criterio cualitativo clave', 'decision': 'evitar leakage y priorizar interpretabilidad en baseline'}
])
display(montaje_experimental)


,componente,decision
0,base principal,dataset_modelado_anual_limpio.csv
1,base complementaria,dataset_operativo_mensual_limpio.csv
2,unidad de analisis,departamento-anio
3,target principal,perdida_rendimiento_anual_pct
4,target complementario,evento_perdida_anual
5,feature set baseline,10 variables parsimoniosas
6,split principal,train 2007-2020 | test 2021-2024
7,split pequeno de verificacion,train 2007-2018 | valid 2019-2020 | test 2021-2024
8,metricas regresion,"MAE, RMSE, R2"
9,metricas clasificacion,"Accuracy, Precision, Recall, F1, matriz de confusion"


## Checklist de salida para Semana 3

Si este notebook queda aprobado, los siguientes notebooks deberian enfocarse solo en entrenar, predecir y reportar resultados.


In [15]:
week3_checklist = pd.DataFrame([
    {'item': 'base oficial de modelado definida', 'estado': 'si'},
    {'item': 'targets definidos', 'estado': 'si'},
    {'item': 'variables excluidas por leakage documentadas', 'estado': 'si'},
    {'item': 'feature set baseline definido', 'estado': 'si'},
    {'item': 'split temporal definido', 'estado': 'si'},
    {'item': 'metricas definidas', 'estado': 'si'},
    {'item': 'estructura minima de reporte definida', 'estado': 'si'},
    {'item': 'listo para notebook 10 de regresion', 'estado': 'si'},
    {'item': 'listo para notebook 11 de clasificacion', 'estado': 'si'}
])
display(week3_checklist)
print('Notebook 09 listo.')


,item,estado
0,base oficial de modelado definida,si
1,targets definidos,si
2,variables excluidas por leakage documentadas,si
3,feature set baseline definido,si
4,split temporal definido,si
5,metricas definidas,si
6,estructura minima de reporte definida,si
7,listo para notebook 10 de regresion,si
8,listo para notebook 11 de clasificacion,si


Notebook 09 listo.
